In [1]:
import pandas as pd
import numpy as np
data = {"order_id": [1,2,3],"supplier_id": [101,102,103],"delivery_date": ["2026-05-01","2026-05-10","2026-05-14"]}
df = pd.DataFrame(data)
print(df)

   order_id  supplier_id delivery_date
0         1          101    2026-05-01
1         2          102    2026-05-10
2         3          103    2026-05-14


In [2]:
df['delivery_date'] = pd.to_datetime(df['delivery_date'])

In [4]:
today = pd.Timestamp.today()
df['delay_days'] = (today - df['delivery_date']).dt.days

In [5]:
df['is_delayed'] = np.where(df['delay_days'] > 0,1,0)

In [6]:
df = df.dropna()

In [7]:
print(df.head())

   order_id  supplier_id delivery_date  delay_days  is_delayed
0         1          101    2026-05-01          13           1
1         2          102    2026-05-10           4           1
2         3          103    2026-05-14           0           0


In [8]:
from pyspark.sql import SparkSession
spark = SparkSession.builder .appName("SupplyChainProcessing") .getOrCreate()

In [10]:
df.to_csv("orders.csv", index=False)
orders_df = spark.read.csv("orders.csv",header=True,inferSchema=True)

In [12]:
from pyspark.sql.functions import col
delayed_df = orders_df.filter(col("is_delayed") == 1)

In [14]:
grouped_df = delayed_df.groupBy("supplier_id").count()
grouped_df.show()

+-----------+-----+
|supplier_id|count|
+-----------+-----+
|        101|    1|
|        102|    1|
+-----------+-----+



In [15]:
grouped_df.write.mode("overwrite") .csv("output/delayed_orders")